# Telco Customer Churn Prediction
**Platform:** Google Colab | **Domain:** Business | **Type:** Binary Classification

---
### Instructions
1. Run **Cell 1** — mounts Google Drive and creates folder structure
2. Run **Cell 2** — installs required libraries
3. Run all remaining cells **top to bottom**
---

## SETUP

In [ ]:
# CELL 1 — Mount Google Drive + create project structure
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/telco-churn'

folders = [
    f'{BASE}/data',
    f'{BASE}/notebooks',
    f'{BASE}/outputs/figures'
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print('Google Drive mounted!')
print(f'Project root: {BASE}')
print('Folders created:')
for f in folders:
    print(f'  {f}')

In [ ]:
# CELL 2 — Install libraries not pre-installed in Colab
!pip install xgboost shap imbalanced-learn --quiet
print('Libraries installed: xgboost, shap, imbalanced-learn')

## PHASE 2 — Load & First Look

In [ ]:
# CELL 3 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
import urllib.request
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('All libraries loaded!')

In [ ]:
# CELL 4 — Download dataset into Google Drive
RAW_DATA = f'{BASE}/data/telco_churn.csv'

if not os.path.exists(RAW_DATA):
    url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
    urllib.request.urlretrieve(url, RAW_DATA)
    print(f'Dataset downloaded to {RAW_DATA}')
else:
    print(f'Dataset already exists — skipping download')

In [ ]:
# CELL 5 — Load and first look
df = pd.read_csv(RAW_DATA)
print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# CELL 6 — Column info and nulls
print('Column info:')
df.info()
print('\nNull values per column:')
print(df.isnull().sum())

In [ ]:
# CELL 7 — Target variable distribution
print('Churn distribution:')
print(df['Churn'].value_counts())
print('\nChurn rate:', df['Churn'].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(6, 4))
fig.patch.set_facecolor('white')
df['Churn'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Churn Distribution', fontweight='bold')
ax.set_xlabel('Churn')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/01_churn_distribution.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# CELL 8 — Statistical summary
print('Numeric columns summary:')
display(df.describe())

In [ ]:
# CELL 9 — Spot the TotalCharges bug
print('TotalCharges dtype:', df['TotalCharges'].dtype)
print('\nSample values:', df['TotalCharges'].sample(5).values)
bad_rows = pd.to_numeric(df['TotalCharges'], errors='coerce').isnull().sum()
print(f'\nRows with bad TotalCharges: {bad_rows}')
print('Finding: TotalCharges stored as string — fix in Phase 3')

## PHASE 3 — Data Cleaning & Feature Engineering

In [ ]:
# CELL 10 — Fix data types + drop useless columns
df_clean = df.copy()

df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
print('NaN in TotalCharges:', df_clean['TotalCharges'].isnull().sum())
print('\nRows with missing TotalCharges (new customers, tenure=0):')
print(df_clean[df_clean['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']])

df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)
df_clean = df_clean.drop(columns=['customerID'])

print('\nShape after cleaning:', df_clean.shape)
print('TotalCharges dtype now:', df_clean['TotalCharges'].dtype)

In [ ]:
# CELL 11 — Understand categorical columns
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Categorical columns:', categorical_cols)
print('\nNumeric columns:', numeric_cols)
for col in categorical_cols:
    print(f'\n{col}: {df_clean[col].unique()}')

In [ ]:
# CELL 12 — Binary encoding (Yes/No -> 1/0)
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_clean[col] = df_clean[col].map(lambda x: 1 if x in ['Yes', 'Male'] else 0)
print('Binary columns encoded:')
print(df_clean[binary_cols].head())

In [ ]:
# CELL 13 — One-hot encoding (multi-category columns)
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                  'Contract', 'PaymentMethod']
df_clean = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True)
print('Shape after one-hot encoding:', df_clean.shape)
print('\nNew columns:')
print([c for c in df_clean.columns if '_' in c])

In [ ]:
# CELL 14 — Feature Engineering (4 domain-informed features)

# Feature 1: Cost relative to loyalty (+1 avoids division by zero)
df_clean['charges_per_tenure'] = df_clean['MonthlyCharges'] / (df_clean['tenure'] + 1)

# Feature 2: New customer flag — captures churn cliff at 6 months
df_clean['is_new_customer'] = (df_clean['tenure'] < 6).astype(int)

# Feature 3: Multiple services proxy — more services = harder to leave
df_clean['has_multiple_services'] = (
    (df_clean['PhoneService'] == 1) &
    (df_clean['MonthlyCharges'] > df_clean['MonthlyCharges'].median())
).astype(int)

# Feature 4: High spend + new = highest risk profile
df_clean['high_spend_new'] = (
    (df_clean['MonthlyCharges'] > 65) &
    (df_clean['tenure'] < 12)
).astype(int)

print('New features added. Sample:')
print(df_clean[['tenure', 'MonthlyCharges', 'charges_per_tenure',
                'is_new_customer', 'has_multiple_services', 'high_spend_new']].head(8))

In [ ]:
# CELL 15 — Save cleaned dataset BEFORE scaling
CLEAN_DATA = f'{BASE}/data/telco_churn_cleaned.csv'
df_clean.to_csv(CLEAN_DATA, index=False)
print(f'Cleaned dataset saved to {CLEAN_DATA}')
print(f'Original: {df.shape} -> Clean: {df_clean.shape}')

In [ ]:
# CELL 16 — Train/Test split BEFORE scaling (prevents data leakage)
from sklearn.model_selection import train_test_split

X = df_clean.drop('Churn', axis=1)
y = df_clean['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape}')
print(f'Test set:      {X_test.shape}')
print(f'Churn rate train: {y_train.mean():.1%}')
print(f'Churn rate test:  {y_test.mean():.1%}')

In [ ]:
# CELL 17 — Scale numeric features (fit on train ONLY)
from sklearn.preprocessing import StandardScaler
import joblib

scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_tenure']
scaler = StandardScaler()
scaler.fit(X_train[scale_cols])

print('Scaler learned from training data:')
for col, mean, std in zip(scale_cols, scaler.mean_, scaler.scale_):
    print(f'  {col:25s} -> mean: {mean:.2f}, std: {std:.2f}')

X_train = X_train.copy()
X_test  = X_test.copy()
X_train[scale_cols] = scaler.transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print('\nAfter scaling (mean~0, std~1):')
print(X_train[scale_cols].describe().loc[['mean', 'std']].round(3))

joblib.dump(scaler, f'{BASE}/outputs/scaler.pkl')
print('\nScaler saved!')

In [ ]:
# CELL 18 — Phase 3 final verification
print('=' * 55)
print('PHASE 3 COMPLETE')
print('=' * 55)
print(f'Clean dataset:    {df_clean.shape}')
print(f'Training set:     {X_train.shape}')
print(f'Test set:         {X_test.shape}')
print(f'Nulls in train:   {X_train.isnull().sum().sum()}')
print(f'Nulls in test:    {X_test.isnull().sum().sum()}')
print(f'Churn rate train: {y_train.mean():.1%}')
print(f'All numeric:      {all(X_train.dtypes != object)}')
print('\nEngineered features in both sets:')
for feat in ['charges_per_tenure', 'is_new_customer', 'has_multiple_services', 'high_spend_new']:
    print(f'  {feat:30s} train:{feat in X_train.columns} | test:{feat in X_test.columns}')

## PHASE 4 — Exploratory Data Analysis (EDA)

In [ ]:
# CELL 19 — EDA Setup
# Reload raw file — need human-readable values for charts, NOT scaled values
df_eda = pd.read_csv(RAW_DATA)
df_eda['TotalCharges'] = pd.to_numeric(df_eda['TotalCharges'], errors='coerce').fillna(0)
df_eda = df_eda.drop(columns=['customerID'])
df_eda['Churn'] = df_eda['Churn'].map(lambda x: 1 if x == 'Yes' else 0)
df_eda['SeniorCitizen'] = df_eda['SeniorCitizen'].astype(int)

# Sanity check — must show real dollar amounts
print('MonthlyCharges range:', df_eda['MonthlyCharges'].min(), 'to', df_eda['MonthlyCharges'].max())
print('Tenure range:', df_eda['tenure'].min(), 'to', df_eda['tenure'].max())
print('Shape:', df_eda.shape)

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 12

In [ ]:
# CELL 20 — Q1: How bad is the churn problem?
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('white')
churn_counts = df_eda['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=colors, width=0.5)
axes[0].set_title('Customer Churn Count', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_axisbelow(True)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', color=colors[i])

axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Rate', fontweight='bold')

plt.suptitle('Q1: Overall Churn Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/02_churn_overview.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Finding: {df_eda["Churn"].mean():.1%} of customers churned.')

In [ ]:
# CELL 21 — Q2: Do newer customers churn more?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

df_eda[df_eda['Churn'] == 0]['tenure'].hist(bins=30, ax=axes[0], alpha=0.7, color='#2ecc71', label='No Churn')
df_eda[df_eda['Churn'] == 1]['tenure'].hist(bins=30, ax=axes[0], alpha=0.7, color='#e74c3c', label='Churn')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[0].legend()
axes[0].grid(linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_axisbelow(True)

df_eda['tenure_group'] = pd.cut(df_eda['tenure'], bins=[0, 6, 12, 24, 48, 72],
                                labels=['0-6mo', '6-12mo', '12-24mo', '24-48mo', '48-72mo'])
churn_by_tenure = df_eda.groupby('tenure_group', observed=True)['Churn'].mean() * 100
bar_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
bars = axes[1].bar(churn_by_tenure.index, churn_by_tenure.values, color=bar_colors)
axes[1].set_xlabel('Tenure Group')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[1].axhline(y=df_eda['Churn'].mean() * 100, color='black', linestyle='--', linewidth=1.5, label='Overall avg')
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
axes[1].set_axisbelow(True)
for bar, val in zip(bars, churn_by_tenure.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10, color='#2c3e50')

plt.suptitle('Q2: Does Tenure Predict Churn?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/03_tenure_churn.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Finding: First 6 months churn at {churn_by_tenure.iloc[0]:.1f}% vs overall {df_eda["Churn"].mean()*100:.1f}%')

In [ ]:
# CELL 22 — Q3: Does contract type predict churn?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

contract_churn = df_eda.groupby('Contract')['Churn'].mean() * 100
bar_colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = axes[0].bar(contract_churn.index, contract_churn.values, color=bar_colors, width=0.5)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0].grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_axisbelow(True)
for bar, val in zip(bars, contract_churn.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', color='#2c3e50')

churn_pivot = df_eda.groupby(['Contract', 'Churn']).size().unstack()
churn_pivot.plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71', '#e74c3c'], width=0.5)
axes[1].set_xlabel('')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Volume by Contract Type', fontweight='bold')
axes[1].legend(['Retained', 'Churned'])
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
axes[1].set_axisbelow(True)

plt.suptitle('Q3: Contract Type vs Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/04_contract_churn.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Finding: Month-to-month churn = {contract_churn.iloc[1]:.1f}% vs Two-year = {contract_churn.iloc[2]:.1f}%')

In [ ]:
# CELL 23 — Q4: Are churners paying more?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

df_eda.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0], patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7),
               medianprops=dict(color='black', linewidth=2))
axes[0].set_xticklabels(['No Churn', 'Churn'])
axes[0].set_xlabel('')
axes[0].set_ylabel('Monthly Charges ($)')
plt.sca(axes[0])
plt.title('Monthly Charges by Churn Status', fontweight='bold')
axes[0].grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_axisbelow(True)

df_eda[df_eda['Churn'] == 0]['MonthlyCharges'].plot(kind='kde', ax=axes[1], color='#2ecc71', linewidth=2, label='No Churn')
df_eda[df_eda['Churn'] == 1]['MonthlyCharges'].plot(kind='kde', ax=axes[1], color='#e74c3c', linewidth=2, label='Churn')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Density')
axes[1].set_title('Charge Distribution — Churn vs Retained', fontweight='bold')
axes[1].legend()
axes[1].axvline(df_eda[df_eda['Churn'] == 1]['MonthlyCharges'].median(), color='#e74c3c', linestyle='--', alpha=0.7)
axes[1].grid(linestyle='--', linewidth=0.5, alpha=0.5)
axes[1].set_axisbelow(True)

plt.suptitle('Q4: Do Churners Pay More?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/05_charges_churn.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()

c_med = df_eda[df_eda['Churn'] == 1]['MonthlyCharges'].median()
r_med = df_eda[df_eda['Churn'] == 0]['MonthlyCharges'].median()
print(f'Finding: Churners pay ${c_med:.0f}/mo vs ${r_med:.0f}/mo retained. Difference: ${c_med - r_med:.0f}/mo')

In [ ]:
# CELL 24 — Q5: Which features correlate with churn? (Self-contained)
df_corr = df_eda.copy()
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df_corr[col] = df_corr[col].map(lambda x: 1 if x == 'Yes' else 0)
df_corr['gender'] = df_corr['gender'].map(lambda x: 1 if x == 'Male' else 0)
df_corr = pd.get_dummies(df_corr, drop_first=True)
if 'tenure_group' in df_corr.columns:
    df_corr = df_corr.drop(columns=['tenure_group'])
df_corr = df_corr.apply(pd.to_numeric, errors='coerce').dropna(axis=1)

churn_corr = df_corr.corr()['Churn'].drop('Churn')
churn_corr_sorted = churn_corr.reindex(churn_corr.abs().sort_values(ascending=False).index).head(15)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor('white')

# Left: bar chart
colors_bar = ['#e74c3c' if x > 0 else '#2ecc71' for x in churn_corr_sorted.values]
bars = axes[0].barh(range(len(churn_corr_sorted)), churn_corr_sorted.values,
                    color=colors_bar, height=0.6, edgecolor='white', linewidth=0.5)
clean_labels = (churn_corr_sorted.index
    .str.replace('InternetService_', 'Internet: ', regex=False)
    .str.replace('Contract_', 'Contract: ', regex=False)
    .str.replace('PaymentMethod_', 'Payment: ', regex=False)
    .str.replace('_No internet service', ' (no internet)', regex=False)
    .str.replace('tenure_group_', 'Tenure: ', regex=False))
axes[0].set_yticks(range(len(churn_corr_sorted)))
axes[0].set_yticklabels(clean_labels, fontsize=11)
axes[0].invert_yaxis()
axes[0].axvline(x=0, color='black', linewidth=1.2)
axes[0].set_xlabel('Correlation with Churn', fontsize=12)
axes[0].set_title('Top 15 Features Correlated with Churn', fontweight='bold', fontsize=13, pad=15)
axes[0].xaxis.set_minor_locator(mticker.MultipleLocator(0.05))
axes[0].grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_axisbelow(True)
for bar, val in zip(bars, churn_corr_sorted.values):
    axes[0].text(val + 0.004 if val > 0 else val - 0.004,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val:+.3f}', va='center', ha='left' if val > 0 else 'right',
                 fontsize=9, fontweight='bold',
                 color='#c0392b' if val > 0 else '#27ae60')
axes[0].legend(handles=[
    Patch(facecolor='#e74c3c', label='Increases churn risk'),
    Patch(facecolor='#2ecc71', label='Decreases churn risk')
], loc='lower right', fontsize=10)

# Right: heatmap (lower triangle)
top_features = churn_corr_sorted.head(10).index.tolist() + ['Churn']
corr_matrix = df_corr[top_features].corr()
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

def shorten(label):
    return (label.replace('InternetService_', 'Internet:\n')
                 .replace('Contract_', 'Contract:\n')
                 .replace('PaymentMethod_', 'Payment:\n')
                 .replace('_No internet service', '\n(no internet)')
                 .replace('tenure_group_', 'Tenure:\n'))

short_labels = [shorten(c) for c in corr_matrix.columns]
sns.heatmap(corr_matrix, ax=axes[1], mask=mask, annot=True, fmt='.2f',
            annot_kws={'size': 9, 'weight': 'bold'}, cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.8,
            linecolor='white', cbar_kws={'shrink': 0.75, 'label': 'Correlation'})
axes[1].set_xticklabels(short_labels, rotation=40, ha='right', fontsize=9, rotation_mode='anchor')
axes[1].set_yticklabels(short_labels, rotation=0, fontsize=9)
axes[1].set_title('Correlation Matrix — Top Features', fontweight='bold', fontsize=13, pad=15)

plt.suptitle('Q5: Which Features Drive Churn?', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/06_correlation_heatmap.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Top 5 positive predictors:')
print(churn_corr_sorted[churn_corr_sorted > 0].head(5).to_string())
print('\nTop 5 protective features:')
print(churn_corr_sorted[churn_corr_sorted < 0].head(5).to_string())

In [ ]:
# CELL 25 — Q6: What is the profile of a typical churner?
fig = plt.figure(figsize=(16, 10))
fig.suptitle('The Anatomy of a Churning Customer', fontsize=16, fontweight='bold', y=1.01)
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
senior_churn = df_eda.groupby('SeniorCitizen')['Churn'].mean() * 100
ax1.bar(['Non-Senior', 'Senior'], senior_churn.values, color=['#2ecc71', '#e74c3c'], width=0.5)
ax1.set_title('Senior Citizens', fontweight='bold')
ax1.set_ylabel('Churn Rate (%)')
ax1.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
ax1.set_axisbelow(True)
for i, v in enumerate(senior_churn.values):
    ax1.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold', color='#2c3e50')

ax2 = fig.add_subplot(gs[0, 1])
billing_churn = df_eda.groupby('PaperlessBilling')['Churn'].mean() * 100
ax2.bar(['Paper Billing', 'Paperless'], billing_churn.values, color=['#2ecc71', '#e74c3c'], width=0.5)
ax2.set_title('Billing Method', fontweight='bold')
ax2.set_ylabel('Churn Rate (%)')
ax2.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
ax2.set_axisbelow(True)
for i, v in enumerate(billing_churn.values):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold', color='#2c3e50')

ax3 = fig.add_subplot(gs[0, 2])
partner_churn = df_eda.groupby('Partner')['Churn'].mean() * 100
ax3.bar(partner_churn.index, partner_churn.values, color=['#e74c3c', '#2ecc71'], width=0.5)
ax3.set_title('Partner Status', fontweight='bold')
ax3.set_ylabel('Churn Rate (%)')
ax3.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
ax3.set_axisbelow(True)
for i, v in enumerate(partner_churn.values):
    ax3.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold', color='#2c3e50')

ax4 = fig.add_subplot(gs[1, 0])
internet_churn = df_eda.groupby('InternetService')['Churn'].mean() * 100
ax4.bar(internet_churn.index, internet_churn.values, color=['#3498db', '#e74c3c', '#2ecc71'], width=0.5)
ax4.set_title('Internet Service', fontweight='bold')
ax4.set_ylabel('Churn Rate (%)')
ax4.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5)
ax4.set_axisbelow(True)
for i, v in enumerate(internet_churn.values):
    ax4.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold', color='#2c3e50')

ax5 = fig.add_subplot(gs[1, 1:])
scatter = ax5.scatter(df_eda['tenure'], df_eda['MonthlyCharges'],
                      c=df_eda['Churn'], cmap='RdYlGn_r', alpha=0.3, s=15)
ax5.set_xlabel('Tenure (months)')
ax5.set_ylabel('Monthly Charges ($)')
ax5.set_title('Tenure vs Charges — Red = churned', fontweight='bold')
plt.colorbar(scatter, ax=ax5, label='Churned')

plt.savefig(f'{BASE}/outputs/figures/07_churner_profile.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Phase 4 complete — all 6 charts saved!')

## PHASE 5 — Model Building

In [ ]:
# CELL 26 — Apply SMOTE (training set ONLY)
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('Before SMOTE:')
print(f'  No churn: {sum(y_train == 0)} | Churn: {sum(y_train == 1)}')
print(f'  Churn rate: {y_train.mean():.1%}')
print('\nAfter SMOTE:')
print(f'  No churn: {sum(y_train_bal == 0)} | Churn: {sum(y_train_bal == 1)}')
print(f'  Churn rate: {y_train_bal.mean():.1%}')
print(f'\nTraining rows: {len(y_train)} -> {len(y_train_bal)}')
print(f'Test set unchanged: {len(y_test)} rows')

In [ ]:
# CELL 27 — Train Random Forest (Baseline)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score,
                             precision_score, recall_score)

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=4,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train_bal, y_train_bal)

rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

print('Random Forest Results:')
print('=' * 45)
print(classification_report(y_test, rf_preds, target_names=['No Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, rf_proba):.4f}')

In [ ]:
# CELL 28 — Train XGBoost (Challenger)
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=sum(y_train == 0) / sum(y_train == 1),
    eval_metric='logloss', random_state=42, n_jobs=-1
)
xgb_model.fit(X_train_bal, y_train_bal, eval_set=[(X_test, y_test)], verbose=False)

xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print('XGBoost Results:')
print('=' * 45)
print(classification_report(y_test, xgb_preds, target_names=['No Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, xgb_proba):.4f}')

In [ ]:
# CELL 29 — Model comparison charts
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('white')

for idx, (model_name, preds) in enumerate([('Random Forest', rf_preds), ('XGBoost', xgb_preds)]):
    ax = axes[idx]
    cm = confusion_matrix(y_test, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
    sns.heatmap(cm_pct, ax=ax, annot=False, cmap='RdYlGn', vmin=0, vmax=100,
                linewidths=2, linecolor='white', cbar_kws={'shrink': 0.75, 'label': '%'})
    labels = [['TN', 'FP'], ['FN', 'TP']]
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.35, labels[i][j], ha='center', va='center',
                    fontsize=11, fontweight='bold', color='white')
            ax.text(j + 0.5, i + 0.6, f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=10, color='white')
    ax.set_xticklabels(['Predicted\nNo Churn', 'Predicted\nChurn'], fontsize=10)
    ax.set_yticklabels(['Actual\nNo Churn', 'Actual\nChurn'], fontsize=10, rotation=0)
    ax.set_title(f'{model_name}\nConfusion Matrix', fontweight='bold', fontsize=12)

ax = axes[2]
for name, proba, color in [('Random Forest', rf_proba, '#3498db'), ('XGBoost', xgb_proba, '#e74c3c')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name}  (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random guess')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_axisbelow(True)

plt.suptitle('Phase 5: Model Evaluation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/08_model_comparison.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# CELL 30 — Model comparison summary table
results = {
    'Model': ['Random Forest', 'XGBoost'],
    'Precision': [precision_score(y_test, rf_preds), precision_score(y_test, xgb_preds)],
    'Recall':    [recall_score(y_test, rf_preds),    recall_score(y_test, xgb_preds)],
    'F1 Score':  [f1_score(y_test, rf_preds),        f1_score(y_test, xgb_preds)],
    'AUC-ROC':   [roc_auc_score(y_test, rf_proba),   roc_auc_score(y_test, xgb_proba)]
}
results_df = pd.DataFrame(results).set_index('Model')
print('=' * 55)
print('MODEL COMPARISON SUMMARY')
print('=' * 55)
print(results_df.round(4).to_string())
print('\nWinner -> highest F1 + AUC combined')

best_model = xgb_model
best_proba = xgb_proba
best_preds = xgb_preds

In [ ]:
# CELL 31 — SHAP Explainability
import shap

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor('white')

# Left: Global feature importance
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns).sort_values(ascending=True).tail(15)
clean_shap_labels = (mean_shap.index
    .str.replace('Contract_', 'Contract: ', regex=False)
    .str.replace('InternetService_', 'Internet: ', regex=False)
    .str.replace('PaymentMethod_', 'Payment: ', regex=False)
    .str.replace('_No internet service', ' (no internet)', regex=False))
colors_shap = ['#e74c3c' if v > mean_shap.median() else '#3498db' for v in mean_shap.values]
bars = axes[0].barh(range(len(mean_shap)), mean_shap.values, color=colors_shap,
                    height=0.65, edgecolor='white', linewidth=0.5)
axes[0].set_yticks(range(len(mean_shap)))
axes[0].set_yticklabels(clean_shap_labels, fontsize=10)
axes[0].set_xlabel('Mean |SHAP value|', fontsize=11)
axes[0].set_title('Top 15 Features by SHAP Importance', fontweight='bold', fontsize=13)
axes[0].grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_axisbelow(True)
for bar, val in zip(bars, mean_shap.values):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold', color='#c0392b')
axes[0].legend(handles=[
    Patch(facecolor='#e74c3c', label='High impact'),
    Patch(facecolor='#3498db', label='Lower impact')
], fontsize=10)

# Right: Single high-risk customer explanation
high_risk_idx = np.argmax(best_proba)
shap_single   = shap_values[high_risk_idx]
feat_names_clean = (pd.Index(X_test.columns)
    .str.replace('Contract_', 'Contract: ', regex=False)
    .str.replace('InternetService_', 'Internet: ', regex=False)
    .str.replace('PaymentMethod_', 'Payment: ', regex=False)
    .str.replace('_No internet service', ' (no internet)', regex=False))
top10_idx   = np.argsort(np.abs(shap_single))[-10:]
top10_shap  = shap_single[top10_idx]
top10_names = feat_names_clean[top10_idx]
colors_single = ['#e74c3c' if v > 0 else '#2ecc71' for v in top10_shap]
axes[1].barh(range(10), top10_shap, color=colors_single, height=0.65, edgecolor='white')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(top10_names, fontsize=10)
axes[1].axvline(x=0, color='black', linewidth=1.2)
axes[1].set_xlabel('SHAP value — push toward churn (+) or away (-)', fontsize=11)
axes[1].set_title(
    f'Why Customer #{high_risk_idx} is High Risk\nPredicted churn probability: {best_proba[high_risk_idx]:.1%}',
    fontweight='bold', fontsize=12)
axes[1].grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.5)
axes[1].set_axisbelow(True)
axes[1].legend(handles=[
    Patch(facecolor='#e74c3c', label='Pushes toward churn'),
    Patch(facecolor='#2ecc71', label='Pushes away from churn')
], fontsize=10)

plt.suptitle('Phase 5: SHAP Explainability', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(pad=2.5)
plt.savefig(f'{BASE}/outputs/figures/09_shap_analysis.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()

print(f'\nCustomer #{high_risk_idx} churn probability: {best_proba[high_risk_idx]:.1%}')
print('\nTop reasons pushing toward churn:')
push = [(top10_names[i], top10_shap[i]) for i in range(10) if top10_shap[i] > 0]
for feat, val in sorted(push, key=lambda x: -x[1])[:5]:
    print(f'  {feat:35s} +{val:.3f}')
print('\nTop reasons pushing away from churn:')
stay = [(top10_names[i], top10_shap[i]) for i in range(10) if top10_shap[i] < 0]
for feat, val in sorted(stay, key=lambda x: x[1])[:3]:
    print(f'  {feat:35s} {val:.3f}')

In [ ]:
# CELL 32 — Save all models and results
import joblib

joblib.dump(rf_model,   f'{BASE}/outputs/random_forest_model.pkl')
joblib.dump(xgb_model,  f'{BASE}/outputs/xgboost_model.pkl')
joblib.dump(explainer,  f'{BASE}/outputs/shap_explainer.pkl')
results_df.round(4).to_csv(f'{BASE}/outputs/model_results.csv')

print('All files saved to Google Drive:')
print(f'  {BASE}/outputs/random_forest_model.pkl')
print(f'  {BASE}/outputs/xgboost_model.pkl')
print(f'  {BASE}/outputs/shap_explainer.pkl')
print(f'  {BASE}/outputs/scaler.pkl')
print(f'  {BASE}/outputs/model_results.csv')
print(f'  {BASE}/outputs/figures/  (9 charts)')

In [ ]:
# CELL 33 — Final Project Summary
print('=' * 60)
print('PROJECT COMPLETE — TELCO CHURN PREDICTION')
print('=' * 60)
xgb_f1  = f1_score(y_test, xgb_preds)
xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_rec = recall_score(y_test, xgb_preds)
xgb_pre = precision_score(y_test, xgb_preds)
print(f'\nFINAL MODEL SCORES (XGBoost):')
print(f'  Precision : {xgb_pre:.3f}')
print(f'  Recall    : {xgb_rec:.3f}')
print(f'  F1 Score  : {xgb_f1:.3f}')
print(f'  AUC-ROC   : {xgb_auc:.3f}')
print('\nKEY BUSINESS INSIGHTS:')
print(f'  1. {df_eda["Churn"].mean():.1%} churn rate — significant retention problem')
print('  2. Month-to-month contracts = highest churn risk')
print('  3. First 6 months = highest risk window')
print('  4. Churners pay more per month than retained customers')
print('  5. SHAP explains every prediction individually')
print('\nREADY FOR GITHUB!')

---
## Project Complete
All outputs saved to `telco-churn/` in your Google Drive.

**Next step:** Download the notebook from Colab → File → Download → Download .ipynb  
Then push everything to GitHub using the README and requirements.txt provided.

---